In [94]:
import pandas as pd
import numpy as np
import torch
import pickle
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

df_master = pd.read_csv('nba_master_dataset.csv', parse_dates=['GAME_DATE'])

print(f"Dataset loaded: {len(df_master):,} rows")
print(f"Columns: {len(df_master.columns)}")

Dataset loaded: 56,985 rows
Columns: 85


In [96]:
# Convert string columns

# Convert HOME_AWAY and POSITION to numeric
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_master['POSITION']  = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

print(f"HOME_AWAY dtype: {df_master['HOME_AWAY'].dtype}")
print(f"POSITION dtype:  {df_master['POSITION'].dtype}")
print()
print(df_master[['PLAYER_NAME', 'HOME_AWAY', 'POSITION']].head(5))

HOME_AWAY dtype: int64
POSITION dtype:  int64

  PLAYER_NAME  HOME_AWAY  POSITION
0    AJ Green          0         0
1    AJ Green          0         0
2    AJ Green          1         0
3    AJ Green          1         0
4    AJ Green          0         0


In [98]:
# Define feature and target columns

import torch.nn as nn

target_cols  = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

feature_cols = [
    # 5-game rolling averages
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',

    # 10-game rolling averages
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',

    # Situational context
    'HOME_AWAY',

    # Round 1
    'DAYS_REST',
    'DEF_RATING',
    'PACE',

    # Round 2
    'OPP_PTS_ALLOWED_PG',
    'OPP_REB_ALLOWED_PG',
    'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG',
    'OPP_STL_PG',
    'OPP_3PM_ALLOWED_PG',

    # Round 3
    'USG_PCT',
    'OPP_PTS_VS_POS',
    'OPP_REB_VS_POS',
    'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS',
    'OPP_STL_VS_POS',
    'OPP_3PM_VS_POS',

    # New rolling features — position normalized
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm',
    'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm',
    'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm',
    'OPP_3PM_VS_POS_roll5_norm',

    # Playoff flag
    'IS_PLAYOFF',

    # Team usage context
    'RELATIVE_USG',
    'USG_RANK',

    # Volatility features
    'PTS_std_roll10',
    'REB_std_roll10',
    'PTS_cv_roll10',
    'REB_cv_roll10',
]

print(f"Total features: {len(feature_cols)}")
print(f"Target outputs: {len(target_cols)}")

Total features: 51
Target outputs: 6


In [100]:
# Add position normalized rolling features and refit scalar

from sklearn.preprocessing import StandardScaler
import pickle

# Add position-normalized rolling features to df_master
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]

for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Refit scaler on raw data
df_raw = df_master.dropna(subset=feature_cols).reset_index(drop=True)

scaler = StandardScaler()
scaler.fit(df_raw[feature_cols])

# Verify
test       = df_raw[feature_cols].iloc[0:1].values
normalized = scaler.transform(test)

print(f"Raw PTS_roll5:        {test[0][0]:.4f}")
print(f"Normalized PTS_roll5: {normalized[0][0]:.4f}")
print(f"Scaler mean PTS_roll5: {scaler.mean_[0]:.4f}")
print()

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Corrected scaler saved")

Raw PTS_roll5:        10.2000
Normalized PTS_roll5: -0.8050
Scaler mean PTS_roll5: 15.7261

✅ Corrected scaler saved


In [102]:
# Build and load model

class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),
        )
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 2)
            )
            for stat in target_stats
        })

    def forward(self, x):
        shared = self.trunk(x)
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = raw[:, 1]
            log_sigma = torch.clamp(log_sigma, min=-3, max=3)
            sigma     = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs

# Load saved weights
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PlayerPropModel(input_dim=len(feature_cols), target_stats=target_cols)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

print(f"Model loaded ✅")
print(f"Device: {device}")
print(f"Input features: {len(feature_cols)}")

Model loaded ✅
Device: cpu
Input features: 51


In [106]:
# Prediction function

def predict_player(player_name, df_master, model, scaler, feature_cols, target_cols):
    """
    Returns predicted (mu, sigma) for each stat using most recent game data.
    Falls back to most recent complete game if latest has NaN features.
    """
    player_df = df_master[df_master['PLAYER_NAME'] == player_name].copy()

    if len(player_df) == 0:
        print(f"❌ Player '{player_name}' not found in dataset")
        return None

    player_df = player_df.sort_values('GAME_DATE')

    # Try most recent game first — if it has NaNs fall back to
    # most recent game with complete features
    latest = player_df.iloc[-1]

    if player_df[feature_cols].iloc[-1].isna().any():
        # Fall back to most recent row with no NaNs in feature cols
        complete_rows = player_df.dropna(subset=feature_cols)
        if len(complete_rows) == 0:
            print(f"❌ No complete feature rows found for {player_name}")
            return None
        latest = complete_rows.iloc[-1]
        print(f"ℹ️  Using last complete game: {latest['GAME_DATE'].date()} "
              f"(most recent has missing opponent stats)")

    game_count = len(player_df)
    if game_count < 50:
        print(f"⚠️  Low confidence — only {game_count} games in dataset")

    features = latest[feature_cols].values.astype(np.float32).reshape(1, -1)
    features = scaler.transform(features)

    if np.isnan(features).any():
        print(f"❌ NaN values in features for {player_name}")
        return None

    feature_tensor = torch.tensor(features, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(feature_tensor)

    results = {}
    for stat in target_cols:
        mu    = outputs[stat][0].item()
        sigma = outputs[stat][1].item()
        results[stat] = {'mu': mu, 'sigma': sigma}

    return results, latest['GAME_DATE'], game_count


# Test on Wembanyama
results, last_game, count = predict_player(
    'Victor Wembanyama', df_master, model, scaler, feature_cols, target_cols
)

print(f"Predictions based on data through: {last_game.date()}")
print(f"Games in dataset: {count}")
print()
print(f"{'Stat':<8} {'Pred μ':>8} {'Pred σ':>8} {'Range (68%)':>20}")
print("-" * 50)
for stat, vals in results.items():
    mu    = vals['mu']
    sigma = vals['sigma']
    low   = mu - sigma
    high  = mu + sigma
    print(f"{stat:<8} {mu:>8.1f} {sigma:>8.1f}   {low:.1f} – {high:.1f}")

Predictions based on data through: 2025-02-12
Games in dataset: 107

Stat       Pred μ   Pred σ          Range (68%)
--------------------------------------------------
PTS          21.7      7.3   14.3 – 29.0
REB           9.4      3.7   5.7 – 13.1
AST           2.7      1.9   0.8 – 4.6
BLK           1.6      1.4   0.2 – 3.0
STL           0.8      1.0   -0.2 – 1.7
FG3M          1.8      1.6   0.1 – 3.4


In [108]:
# EV calculator

def calculate_ev(player_name, stat, prop_line, over_juice, under_juice,
                 df_master, model, scaler, feature_cols, target_cols):
    """
    Calculates EV of betting over or under a prop line.
    Juice format: American odds e.g. -110, +120
    """
    result = predict_player(
        player_name, df_master, model, scaler, feature_cols, target_cols
    )

    if result is None:
        return None

    predictions, last_game, game_count = result

    if stat not in predictions:
        print(f"❌ Stat '{stat}' not available")
        return None

    mu    = predictions[stat]['mu']
    sigma = predictions[stat]['sigma']

    prob_over  = 1 - norm.cdf(prop_line, mu, sigma)
    prob_under = norm.cdf(prop_line, mu, sigma)

    def american_to_prob(juice):
        if juice < 0:
            return abs(juice) / (abs(juice) + 100)
        else:
            return 100 / (juice + 100)

    breakeven_over  = american_to_prob(over_juice)
    breakeven_under = american_to_prob(under_juice)

    edge_over  = prob_over  - breakeven_over
    edge_under = prob_under - breakeven_under

    if edge_over > edge_under and edge_over > 0:
        recommendation = 'OVER'
        edge = edge_over
    elif edge_under > edge_over and edge_under > 0:
        recommendation = 'UNDER'
        edge = edge_under
    else:
        recommendation = 'NO BET'
        edge = max(edge_over, edge_under)

    print(f"{'='*50}")
    print(f"  {player_name} — {stat} Prop Analysis")
    print(f"{'='*50}")
    print(f"  Data through:     {last_game.date()}")
    print(f"  Games in dataset: {game_count}")
    print()
    print(f"  Model prediction: μ={mu:.1f}  σ={sigma:.1f}")
    print(f"  Prop line:        {prop_line}")
    print()
    print(f"  Model P(over):    {prob_over:.1%}")
    print(f"  Model P(under):   {prob_under:.1%}")
    print()
    print(f"  Breakeven over:   {breakeven_over:.1%}  (juice: {over_juice})")
    print(f"  Breakeven under:  {breakeven_under:.1%}  (juice: {under_juice})")
    print()
    print(f"  Edge over:        {edge_over:+.1%}")
    print(f"  Edge under:       {edge_under:+.1%}")
    print()

    if recommendation == 'NO BET':
        print(f"  🚫 NO BET — no meaningful edge found")
    else:
        print(f"  ✅ BET {recommendation} {prop_line} {stat}")
        print(f"  Edge: {edge:+.1%}")

    print(f"{'='*50}")

    return {
        'player':         player_name,
        'stat':           stat,
        'line':           prop_line,
        'mu':             mu,
        'sigma':          sigma,
        'prob_over':      prob_over,
        'prob_under':     prob_under,
        'edge_over':      edge_over,
        'edge_under':     edge_under,
        'recommendation': recommendation,
        'edge':           edge,
        'game_count':     game_count
    }


# Test — Wemby points prop
result = calculate_ev(
    player_name  = 'Victor Wembanyama',
    stat         = 'PTS',
    prop_line    = 24.5,
    over_juice   = -110,
    under_juice  = -110,
    df_master    = df_master,
    model        = model,
    scaler       = scaler,
    feature_cols = feature_cols,
    target_cols  = target_cols
)

  Victor Wembanyama — PTS Prop Analysis
  Data through:     2025-02-12
  Games in dataset: 107

  Model prediction: μ=21.7  σ=7.3
  Prop line:        24.5

  Model P(over):    35.0%
  Model P(under):   65.0%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -17.4%
  Edge under:       +12.7%

  ✅ BET UNDER 24.5 PTS
  Edge: +12.7%


In [110]:
# Test across multiple players and stats
test_cases = [
    ('Nikola Jokić',              'PTS',  27.5, -110, -110),
    ('Nikola Jokić',              'REB',  12.5, -115, -105),
    ('Stephen Curry',             'FG3M',  4.5, -120, +100),
    ('Jayson Tatum',              'PTS',  28.5, -110, -110),
    ('Giannis Antetokounmpo',     'REB',  11.5, -110, -110),
    ('Shai Gilgeous-Alexander',   'PTS',  31.5, -110, -110),
]

print("PROP SCAN RESULTS")
print("=" * 70)

for player, stat, line, over_j, under_j in test_cases:
    result = calculate_ev(
        player_name  = player,
        stat         = stat,
        prop_line    = line,
        over_juice   = over_j,
        under_juice  = under_j,
        df_master    = df_master,
        model        = model,
        scaler       = scaler,
        feature_cols = feature_cols,
        target_cols  = target_cols
    )
    print()

PROP SCAN RESULTS
ℹ️  Using last complete game: 2025-04-13 (most recent has missing opponent stats)
  Nikola Jokić — PTS Prop Analysis
  Data through:     2025-04-13
  Games in dataset: 375

  Model prediction: μ=26.4  σ=7.7
  Prop line:        27.5

  Model P(over):    44.3%
  Model P(under):   55.7%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -8.1%
  Edge under:       +3.3%

  ✅ BET UNDER 27.5 PTS
  Edge: +3.3%

ℹ️  Using last complete game: 2025-04-13 (most recent has missing opponent stats)
  Nikola Jokić — REB Prop Analysis
  Data through:     2025-04-13
  Games in dataset: 375

  Model prediction: μ=10.5  σ=3.9
  Prop line:        12.5

  Model P(over):    30.4%
  Model P(under):   69.6%

  Breakeven over:   53.5%  (juice: -115)
  Breakeven under:  51.2%  (juice: -105)

  Edge over:        -23.1%
  Edge under:       +18.4%

  ✅ BET UNDER 12.5 REB
  Edge: +18.4%

ℹ️  Using last complete game: 2025-04-13 (most recent has m

In [112]:
# Test with lines closer to regular season averages
test_cases_regular = [
    ('Nikola Jokić',            'PTS',  25.5, -110, -110),
    ('Nikola Jokić',            'REB',  11.5, -110, -110),
    ('Stephen Curry',           'FG3M',  3.5, -110, -110),
    ('Jayson Tatum',            'PTS',  25.5, -110, -110),
    ('Victor Wembanyama',       'PTS',  22.5, -110, -110),
    ('Victor Wembanyama',       'BLK',   2.5, -110, -110),
    ('Shai Gilgeous-Alexander', 'PTS',  28.5, -110, -110),
    ('Giannis Antetokounmpo',   'PTS',  28.5, -110, -110),
]

print("REGULAR SEASON LINE TEST")
print("=" * 50)

for player, stat, line, over_j, under_j in test_cases_regular:
    result = calculate_ev(
        player_name  = player,
        stat         = stat,
        prop_line    = line,
        over_juice   = over_j,
        under_juice  = under_j,
        df_master    = df_master,
        model        = model,
        scaler       = scaler,
        feature_cols = feature_cols,
        target_cols  = target_cols
    )
    print()

REGULAR SEASON LINE TEST
ℹ️  Using last complete game: 2025-04-13 (most recent has missing opponent stats)
  Nikola Jokić — PTS Prop Analysis
  Data through:     2025-04-13
  Games in dataset: 375

  Model prediction: μ=26.4  σ=7.7
  Prop line:        25.5

  Model P(over):    54.6%
  Model P(under):   45.4%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        +2.2%
  Edge under:       -7.0%

  ✅ BET OVER 25.5 PTS
  Edge: +2.2%

ℹ️  Using last complete game: 2025-04-13 (most recent has missing opponent stats)
  Nikola Jokić — REB Prop Analysis
  Data through:     2025-04-13
  Games in dataset: 375

  Model prediction: μ=10.5  σ=3.9
  Prop line:        11.5

  Model P(over):    39.9%
  Model P(under):   60.1%

  Breakeven over:   52.4%  (juice: -110)
  Breakeven under:  52.4%  (juice: -110)

  Edge over:        -12.5%
  Edge under:       +7.7%

  ✅ BET UNDER 11.5 REB
  Edge: +7.7%

ℹ️  Using last complete game: 2025-04-13 (most recent h